## Data Prep for Multi-Modal RAG 

### Installing Utilities and Libraries

In [1]:
%pip install numpy==1.26.4 openai==1.69.0 langchain-community==0.4.1 PyPDF2==3.0.1 anthropic==0.120.2

Looking in indexes: https://pypi.org/simple/

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: /usr/local/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Setting up the Environment

In [2]:
import os
from dotenv import load_dotenv
import os
import base64
import mimetypes
import pandas as pd

from anthropic import Anthropic
from PyPDF2 import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

claude_model_name = os.getenv("CLAUDE_MODEL_NAME")
claude_api_key = os.getenv("CLAUDE_API_KEY")

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


### Create the Anthropic Client

In [3]:
import anthropic

client = anthropic.Anthropic(api_key=claude_api_key)

### Verify and List the Dataset

In [4]:
knowledge_folder = "./knowledge"
docs_folder = "./knowledge/docs"
images_folder = "./knowledge/images"

print("Knowledge folder:")
print(os.listdir(knowledge_folder))

print("\nImages:")
print(os.listdir(images_folder))

print("\nDocuments:")
print(os.listdir(docs_folder))

Knowledge folder:
['.DS_Store', 'images', 'docs']

Images:
['carbonops_MARK.png', 'carbonops_WAVE.png', 'carbonops_TCFD.png', 'BRSR_Materiality_Journey.png', 'SDG_Mapping.png', 'materiality_dimensions.png']

Documents:
['CarbonOps_Sustainability_Evidence_Schema.pdf', 'CarbonOps_ESG_Intelligence_Model.pdf', 'CarbonOps_Employee_Well_Being.pdf', 'CarbonOps_Global_Remote_Work.pdf']


### Define the Base64 Conversion Function for Images

In [5]:
def encode_image(image_path):
    """Convert an image into a Base64 encoded string."""

    with open(image_path, "rb") as image_file:
        return base64.b64encode(
            image_file.read()
        ).decode("utf-8")

### Process all Images and Create an Array

In [6]:
image_records = []

for filename in os.listdir(images_folder):

    if filename.lower().endswith(".png"):

        image_path = os.path.join(
            images_folder,
            filename
        )

        image_records.append(
            {
                "content_path": image_path,
                "base64_content": encode_image(image_path),
            }
        )


images_metadata = pd.DataFrame(image_records)

print(f"Images processed: {len(images_metadata)}")

images_metadata.head()

Images processed: 6


,content_path,base64_content
0,./knowledge/images/carbonops_MARK.png,iVBORw0KGgoAAAANSUhEUgAABUUAAALyCAIAAABy6CLYAA...
1,./knowledge/images/carbonops_WAVE.png,iVBORw0KGgoAAAANSUhEUgAABUEAAALzCAIAAACwX1EHAA...
2,./knowledge/images/carbonops_TCFD.png,iVBORw0KGgoAAAANSUhEUgAABUQAAAL1CAIAAACAL3leAA...
3,./knowledge/images/BRSR_Materiality_Journey.png,iVBORw0KGgoAAAANSUhEUgAABUMAAALrCAIAAABbLwHMAA...
4,./knowledge/images/SDG_Mapping.png,iVBORw0KGgoAAAANSUhEUgAABT8AAALsCAIAAACqX1RiAA...


### Create the Image Verbalization Function

In [8]:
def verbalize_image(image_path):

    image_data = encode_image(image_path)

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=500,

        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": image_data,
                        },
                    },
                    {
                        "type": "text",
                        "text": (
                            "Describe this image in detail for use "
                            "in a retrieval-augmented generation "
                            "knowledge base. Capture important "
                            "concepts, objects, labels, relationships, "
                            "and textual information."
                        ),
                    },
                ],
            }
        ],
    )

    return response.content[0].text

### Process all Images with Verbalization

In [9]:
image_verbalizations = []

for _, row in images_metadata.iterrows():

    print(f"Processing: {row['content_path']}")

    description = verbalize_image(
        row["content_path"]
    )

    image_verbalizations.append(
        {
            "content_path": row["content_path"],
            "chunk": description,
            "content_type": "image",
        }
    )


images_verbalization = pd.DataFrame(
    image_verbalizations
)

images_verbalization.head()

Processing: ./knowledge/images/carbonops_MARK.png
Processing: ./knowledge/images/carbonops_WAVE.png
Processing: ./knowledge/images/carbonops_TCFD.png
Processing: ./knowledge/images/BRSR_Materiality_Journey.png
Processing: ./knowledge/images/SDG_Mapping.png
Processing: ./knowledge/images/materiality_dimensions.png


,content_path,chunk,content_type
0,./knowledge/images/carbonops_MARK.png,# M.A.R.K for ESRS DMA by CarbonOps — Detailed...,image
1,./knowledge/images/carbonops_WAVE.png,# W.A.V.E. for ESRS by CarbonOps — Detailed Im...,image
2,./knowledge/images/carbonops_TCFD.png,# CarbonOps TCFD Automation Flow — Knowledge B...,image
3,./knowledge/images/BRSR_Materiality_Journey.png,# BRSR Materiality Journey - Knowledge Base En...,image
4,./knowledge/images/SDG_Mapping.png,# SDG Mapping Journey - Knowledge Base Entry\n...,image


### Create Recursive Chunking Function for PDFs

In [10]:
def perform_fixed_size_chunking(
    document,
    chunk_size=2000,
    chunk_overlap=500,
):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,

        separators=[
            "\n\n",
            "\n",
            ". ",
            " ",
            "",
        ],
    )

    return text_splitter.split_text(
        document
    )

### Process PDFs with Chunking

In [11]:
document_chunks = []


for filename in os.listdir(docs_folder):

    file_path = os.path.join(
        docs_folder,
        filename
    )

    if (
        os.path.isfile(file_path)
        and filename.lower().endswith(".pdf")
    ):

        print(
            f"Processing: {filename}"
        )

        reader = PdfReader(
            file_path
        )

        text = ""

        for page in reader.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        # Only process documents
        # containing extracted text
        if text.strip():

            chunks = perform_fixed_size_chunking(
                text
            )

            for i, chunk in enumerate(chunks):

                if chunk.strip():

                    document_chunks.append(
                        {
                            "content_path":
                                file_path,

                            "chunk":
                                chunk,

                            "content_type":
                                "document",
                        }
                    )

Processing: CarbonOps_Sustainability_Evidence_Schema.pdf
Processing: CarbonOps_ESG_Intelligence_Model.pdf
Processing: CarbonOps_Employee_Well_Being.pdf
Processing: CarbonOps_Global_Remote_Work.pdf


In [12]:
docs_chunks = pd.DataFrame(
    document_chunks
)

print(
    f"Total document chunks: "
    f"{len(docs_chunks)}"
)

docs_chunks.head()

Total document chunks: 19


,content_path,chunk,content_type
0,./knowledge/docs/CarbonOps_Sustainability_Evid...,https://www.carbonops.cloud/ \n \nCarbonOps S...,document
1,./knowledge/docs/CarbonOps_Sustainability_Evid...,"sensor outputs, and metered readings documenti...",document
2,./knowledge/docs/CarbonOps_Sustainability_Evid...,supported by this evidence. \n11. Retention D...,document
3,./knowledge/docs/CarbonOps_Sustainability_Evid...,"green for SES -A and SES -M, midnight blue for...",document
4,./knowledge/docs/CarbonOps_Sustainability_Evid...,indicating evidence urgency and review status....,document


### Check number of Chunks per PDF

In [14]:
docs_chunks.groupby(
    "content_path"
).size().reset_index(
    name="chunk_count"
)

,content_path,chunk_count
0,./knowledge/docs/CarbonOps_ESG_Intelligence_Mo...,6
1,./knowledge/docs/CarbonOps_Employee_Well_Being...,3
2,./knowledge/docs/CarbonOps_Global_Remote_Work.pdf,5
3,./knowledge/docs/CarbonOps_Sustainability_Evid...,5


### Combine Image and Document Chunks

In [15]:
final_rag_dataset = pd.concat(
    [
        images_verbalization,
        docs_chunks,
    ],
    ignore_index=True,
)

### Create final RAG Dataset with unique IDs

In [16]:
final_rag_dataset.insert(
    0,
    "id",
    range(1, len(final_rag_dataset) + 1)
)

In [17]:
final_rag_dataset.head()

,id,content_path,chunk,content_type
0,1,./knowledge/images/carbonops_MARK.png,# M.A.R.K for ESRS DMA by CarbonOps — Detailed...,image
1,2,./knowledge/images/carbonops_WAVE.png,# W.A.V.E. for ESRS by CarbonOps — Detailed Im...,image
2,3,./knowledge/images/carbonops_TCFD.png,# CarbonOps TCFD Automation Flow — Knowledge B...,image
3,4,./knowledge/images/BRSR_Materiality_Journey.png,# BRSR Materiality Journey - Knowledge Base En...,image
4,5,./knowledge/images/SDG_Mapping.png,# SDG Mapping Journey - Knowledge Base Entry\n...,image


### Save it to a JSON File

In [19]:
final_rag_dataset.to_json(
    "final_rag_dataset.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("Dataset saved to final_rag_dataset.json")

Dataset saved to final_rag_dataset.json
